In [7]:
import pandas as pd

# Load the 'IMDB Dataset.csv' file into a pandas DataFrame
df = pd.read_csv('/content/sample_data/IMDB Dataset.csv')

# Convert all text in the 'review' column to lowercase
df['review'] = df['review'].str.lower()

# Create a list named positive_keywords (updated to 3 keywords)
positive_keywords = ['great', 'excellent', 'love']

# Create a list named negative_keywords (updated to 3 keywords)
negative_keywords = ['bad', 'terrible', 'awful']

print("DataFrame loaded and 'review' column converted to lowercase.")
print("Positive keywords list initialized.")
print("Negative keywords list initialized.")

df.head()

DataFrame loaded and 'review' column converted to lowercase.
Positive keywords list initialized.
Negative keywords list initialized.


,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [8]:
num_positive_reviews = df[df['sentiment'] == 'positive'].shape[0]
total_reviews = df.shape[0]
p_positive = num_positive_reviews / total_reviews

print(f"Number of positive reviews: {num_positive_reviews}")
print(f"Total number of reviews: {total_reviews}")
print(f"Prior probability P(Positive): {p_positive:.4f}")

Number of positive reviews: 25000
Total number of reviews: 50000
Prior probability P(Positive): 0.5000


In [12]:
def calculate_p_keyword(keyword, df):
    """
    Calculates the marginal probability of a keyword appearing in any review, P(keyword).
    """
    reviews_with_keyword = df[df['review'].str.contains(rf'\b{keyword}\b', regex=True, na=False)].shape[0]
    total_reviews = df.shape[0]
    return reviews_with_keyword / total_reviews

def calculate_p_keyword_given_positive(keyword, df):
    """
    Calculates the likelihood of a keyword appearing given a positive review, P(keyword|Positive).
    """
    positive_reviews = df[df['sentiment'] == 'positive']
    positive_reviews_with_keyword = positive_reviews[positive_reviews['review'].str.contains(rf'\b{keyword}\b', regex=True, na=False)].shape[0]
    total_positive_reviews = positive_reviews.shape[0]

    # Add a small constant to avoid division by zero if no positive reviews exist (though unlikely in this dataset)
    if total_positive_reviews == 0:
        return 0.0
    return positive_reviews_with_keyword / total_positive_reviews

def calculate_p_positive_given_keyword(p_positive, p_keyword, p_keyword_given_positive):
    """
    Computes the posterior probability P(Positive|keyword) using Bayes' Theorem.
    P(Positive|keyword) = [P(keyword|Positive) * P(Positive)] / P(keyword).
    """
    # Add a small constant to P(keyword) to avoid division by zero
    if p_keyword == 0:
        return 0.0
    return (p_keyword_given_positive * p_positive) / p_keyword

print("Bayes' Theorem helper functions defined.")

Bayes' Theorem helper functions defined.


In [13]:
all_keywords = positive_keywords + negative_keywords
results = []

for keyword in all_keywords:
    p_keyword = calculate_p_keyword(keyword, df)
    p_keyword_given_positive = calculate_p_keyword_given_positive(keyword, df)
    p_positive_given_keyword = calculate_p_positive_given_keyword(p_positive, p_keyword, p_keyword_given_positive)

    results.append({
        'Keyword': keyword,
        'Sentiment_Category': 'Positive' if keyword in positive_keywords else 'Negative',
        'P(Positive)': f"{p_positive:.4f}",
        'P(keyword)': f"{p_keyword:.4f}",
        'P(keyword|Positive)': f"{p_keyword_given_positive:.4f}",
        'P(Positive|keyword)': f"{p_positive_given_keyword:.4f}"
    })

df_results = pd.DataFrame(results)
print("### Probabilities for All Keywords\n")
print(df_results.to_markdown(index=False))

### Probabilities for All Keywords

| Keyword   | Sentiment_Category   |   P(Positive) |   P(keyword) |   P(keyword|Positive) |   P(Positive|keyword) |
|:----------|:---------------------|--------------:|-------------:|----------------------:|----------------------:|
| great     | Positive             |           0.5 |       0.2515 |                0.3391 |                0.6741 |
| excellent | Positive             |           0.5 |       0.071  |                0.1147 |                0.8074 |
| love      | Positive             |           0.5 |       0.1785 |                0.2266 |                0.6349 |
| bad       | Negative             |           0.5 |       0.2358 |                0.1184 |                0.2512 |
| terrible  | Negative             |           0.5 |       0.054  |                0.0153 |                0.1418 |
| awful     | Negative             |           0.5 |       0.0577 |                0.0114 |                0.0985 |
